In [4]:
import os
import io
import numpy as np
import scipy.io.wavfile as wav
import librosa
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

print("THIS IS CELL 1: Imports are completely ready!")

THIS IS CELL 1: Imports are completely ready!


In [5]:
def process_audio_to_spectrogram(file_path, target_sr=22050, duration=4):
    try:
        y, sr = librosa.load(file_path, sr=target_sr, duration=duration)
        target_length = target_sr * duration
        if len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        else:
            y = y[:target_length]
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
        mel_db = librosa.power_to_db(mel_spec, ref=np.max)
        return (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
    except:
        return None

print(" THIS IS CELL 2: Preprocessing pipeline is completely ready!")

 THIS IS CELL 2: Preprocessing pipeline is completely ready!


In [12]:
def augment_audio(y, sr):
    """Generate 3 augmented variants of one audio clip"""
    variants = []
    
    # Pitch shift up slightly
    variants.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=2))
    # Pitch shift down slightly
    variants.append(librosa.effects.pitch_shift(y, sr=sr, n_steps=-2))
    # Add light background noise
    noise = np.random.normal(0, 0.005, y.shape)
    variants.append(y + noise)
    
    return variants

In [18]:
#  CELL 3: Local data simulator deactivated so your real voice recordings are protected!

In [20]:
import os
print(" Jupyter is actively looking at this path:")
print(os.getcwd())

print("\n Items in this folder are:")
print(os.listdir(os.getcwd()))

 Jupyter is actively looking at this path:
C:\Users\aanch\Desktop\Audio-Deepfake-Detector

 Items in this folder are:
['.ipynb_checkpoints', 'app.py', 'Audio_Deepfake_Detector.ipynb', 'dataset', 'regional_detector.pth']


In [21]:
X, y = [], []
base_dir = os.getcwd()
categories = {'real': 0, 'fake': 1}
valid_extensions = ('.wav', '.mp3', '.m4a', '.aac', '.ogg')

for folder_name, label in categories.items():
    folder_path = os.path.join(base_dir, "dataset", folder_name)
    if not os.path.exists(folder_path):
        print(f" Warning: Folder missing at {folder_path}")
        continue

    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(valid_extensions):
            file_fullpath = os.path.join(folder_path, file_name)
            
            # Original clip
            spec = process_audio_to_spectrogram(file_fullpath)
            if spec is not None:
                X.append(spec)
                y.append(label)
            
            # Augmented variants — load raw audio once, augment, then convert each to spectrogram
            try:
                raw_y, sr = librosa.load(file_fullpath, sr=22050, duration=4)
                for variant in augment_audio(raw_y, sr):
                    mel_spec = librosa.feature.melspectrogram(y=variant, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
                    mel_db = librosa.power_to_db(mel_spec, ref=np.max)
                    norm_spec = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)
                    X.append(norm_spec)
                    y.append(label)
            except:
                pass

print(f"📊 Total valid audio files loaded (with augmentation): {len(X)}")


📊 Total valid audio files loaded (with augmentation): 48


In [22]:
class DeepFakeAudioCNN(nn.Module):
    def __init__(self):
        super(DeepFakeAudioCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2), 
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2), 
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2), 
            nn.Dropout(0.3)      
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(21504, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 1), nn.Sigmoid() 
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = DeepFakeAudioCNN()
criterion = nn.BCELoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print(" THIS IS CELL 5: CNN Model Architecture initialized successfully!")


      

 THIS IS CELL 5: CNN Model Architecture initialized successfully!


In [30]:
epochs = 15
print(" Starting training loop...")
for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        correct += ((outputs >= 0.5).float() == labels).sum().item()
        total += labels.size(0)
    print(f"Epoch [{epoch+1}/{epochs}] -> Accuracy: {(correct/total)*100:.2f}%")

desktop_path = os.path.join(os.path.expanduser("~"), "Desktop", "regional_detector.pth")
torch.save(model.state_dict(), desktop_path)
print(f"\n THIS IS CELL 6: Model saved cleanly directly to Desktop!")

 Starting training loop...
Epoch [1/15] -> Accuracy: 100.00%
Epoch [2/15] -> Accuracy: 100.00%
Epoch [3/15] -> Accuracy: 88.89%
Epoch [4/15] -> Accuracy: 100.00%
Epoch [5/15] -> Accuracy: 88.89%
Epoch [6/15] -> Accuracy: 100.00%
Epoch [7/15] -> Accuracy: 88.89%
Epoch [8/15] -> Accuracy: 100.00%
Epoch [9/15] -> Accuracy: 100.00%
Epoch [10/15] -> Accuracy: 77.78%
Epoch [11/15] -> Accuracy: 100.00%
Epoch [12/15] -> Accuracy: 100.00%
Epoch [13/15] -> Accuracy: 100.00%
Epoch [14/15] -> Accuracy: 100.00%
Epoch [15/15] -> Accuracy: 100.00%

 THIS IS CELL 6: Model saved cleanly directly to Desktop!


In [31]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        correct += ((outputs >= 0.5).float() == labels).sum().item()
        total += labels.size(0)
print(f"Test Accuracy: {(correct/total)*100:.2f}%")

Test Accuracy: 66.67%


In [29]:
%%writefile app.py
import streamlit as st
import torch
import numpy as np
import librosa
import os
import torch.nn as nn

class DeepFakeAudioCNN(nn.Module):
    def __init__(self):
        super(DeepFakeAudioCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2, 2), 
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2), 
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2), 
            nn.Dropout(0.3)      
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(21504, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, 1), nn.Sigmoid() 
        )
    def forward(self, x):
        return self.classifier(self.features(x))

def process_audio_to_spectrogram(file_path, target_sr=22050, duration=4):
    # librosa.load seamlessly uses soundfile behind the scenes to decode mp3/m4a/wav into standard arrays
    y, sr = librosa.load(file_path, sr=target_sr, duration=duration)
    target_length = target_sr * duration
    if len(y) < target_length:
        y = np.pad(y, (0, target_length - len(y)), mode='constant')
    else:
        y = y[:target_length]
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, n_fft=2048, hop_length=512)
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)
    return (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-6)

st.set_page_config(page_title="AI Audio Detector", page_icon="🛡️")
st.title("🛡️ Regional Language Audio Deepfake Detector")
st.write("Upload a regional voice note below to verify if it is an authentic human or an AI voice clone.")

#  MULTI-FORMAT SUPPORT TRIGGERED HERE:
uploaded_file = st.file_uploader(
    "Upload Audio Note (Voice Message, Call Recording, etc.)", 
    type=["wav", "mp3", "m4a", "aac", "ogg"]
)

if uploaded_file is not None:
    st.audio(uploaded_file, format='audio/wav')
    with st.spinner("Analyzing audio micro-textures..."):
        spec = process_audio_to_spectrogram(uploaded_file)
        spec_tensor = torch.tensor(spec).unsqueeze(0).unsqueeze(0).float()
        
        current_dir = os.path.dirname(os.path.abspath(__file__))
        model_path = os.path.join(current_dir, "regional_detector.pth")
        
        model = DeepFakeAudioCNN()
        model.load_state_dict(torch.load(model_path))
        model.eval()
        
        with torch.no_grad():
            prediction = model(spec_tensor).item()
            
        confidence = prediction if prediction > 0.5 else (1 - prediction)
        percentage = f"{confidence * 100:.2f}%"
        
        st.markdown("---")
        if prediction > 0.5:
            st.error(" **Deepfake Variant Detected!**")
            st.metric(label="AI Probability Clone Score", value=percentage)
        else:
            st.success(" **Authentic Human Voice Verified**")
            st.metric(label="Human Consistency Score", value=percentage)

Overwriting app.py


In [15]:
import os
print("Your app.py is saved at:", os.path.join(os.getcwd(), "app.py"))

Your app.py is saved at: C:\Users\aanch\Desktop\Audio-Deepfake-Detector\app.py
